In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3

### Functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [3]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 08_retro_scoring
Subtask: 09_swap_in_swap_out_plot


### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [5]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tbl1 as\n'
 '\t(\n'
 '\t\tselect DimScoreCard.*,\n'
 '\t\ttbltempstaticpool.bigaccountID,\n'
 '\t\tRow_number() OVER(partition BY intAccountKey ORDER BY dtmStampCreation '
 'desc) RowNum\n'
 '\t\tfrom riskdb.analytics.tbltempstaticpool left outer join '
 'edw.pfsedw.dbo.DimScoreCard \n'
 '\t\t\ton tbltempstaticpool.bigAccountId = DimScoreCard.intAccountKey\n'
 "\t\twhere dtmstampcreation >= '2022-01-01'\n"
 "\t\tand strScoreCardVersion not in ( 'genxii', 'dlv1', 'dlv1_1')\n"
 '\n'
 '\t)\n'
 '\t\tselect\n'
 '\t\tdtmstampcreation as App_date, \n'
 '\t\tbigAccountId, \n'
 '\t\tfltDebtorScore, \n'
 '\t\tstrScoreCardVersion\n'
 '\t\tfrom tbl1 \n'
 '\t\twhere RowNum = 1\n'
 '\t\torder by dtmstampcreation')


### Write into df

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# show
df

Wall time: 22.2 s


,App_date,bigAccountId,fltDebtorScore,strScoreCardVersion
0,2022-01-03 14:47:37.927,5838462,0.141234,genxi
1,2022-01-03 15:01:23.737,5872184,0.148215,genxi
2,2022-01-03 15:17:17.263,5828280,0.277955,genxi
3,2022-01-03 15:26:18.493,5869179,0.218271,genxi
4,2022-01-03 15:33:01.133,5856159,0.093084,genxi
...,...,...,...,...
54698,2024-02-16 22:56:54.567,7537277,0.127174,genxi_v2_2
54699,2024-02-16 22:59:13.010,7559506,0.149535,genxi_v2_2
54700,2024-02-16 23:00:35.170,7562092,0.182683,genxi_v2_2
54701,2024-02-16 23:00:38.560,7565692,0.089162,genxi_v2_2


### Save

In [7]:
%%time

# save
str_filename = 'df_genxi_scores.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 1.37 s


### Upload to s3

In [8]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 1.05 s


### Clean-up

In [9]:
os.remove(str_local_path)